### Structured Output 

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. langChain supports multiple schema types and methods for enforcing structurede output.

### Pydantic

Pydantic models provide teh richest feature set with field validation, descriptions, and nested strutures.

In [38]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3.6-27b",max_tokens = 600, temperature=0)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, client=<groq.resources.chat.completions.Completions object at 0x12c0d5480>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x12c0d5e00>, model_name='qwen/qwen3.6-27b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'), max_tokens=600)

In [39]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This year the movie was released")
    director: str = Field(description="The Director of this Movie")
    ratings: float = Field(description="These are the ratings of this movie out of 10")

In [40]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, client=<groq.resources.chat.completions.Completions object at 0x12c0d5480>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x12c0d5e00>, model_name='qwen/qwen3.6-27b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'), max_tokens=600), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The Director of this Movie', 'type': 'string'}, 'ratings': {'description': 'These are the ratings of this movie out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'ratings'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'ty

In [42]:
response = model_with_structure.invoke("provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', ratings=8.8)

### Message output along side parsed structure

In [43]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(...,description="The title of the movie")
    year: int = Field(...,description="This year the movie was released")
    director: str = Field(...,description="The Director of this Movie")
    ratings: float = Field(...,description="These are the ratings of this movie out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("provide deatils baout the movie Incpetion")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "provide deatils baout the movie Incpetion"\n   - There are typos: "deatils" -> "details", "baout" -> "about", "Incpetion" -> "Inception"\n   - The user wants details about the movie "Inception".\n\n2.  **Identify Required Information:**\n   - The available tool is `Movie` which requires: `title`, `year`, `director`, `ratings`\n   - I need to provide details about "Inception". I know from general knowledge:\n     - Title: Inception\n     - Year: 2010\n     - Director: Christopher Nolan\n     - Ratings: Typically around 8.8/10 on IMDb, but I should use a standard/reliable rating. I\'ll use 8.8 as it\'s widely recognized.\n\n3.  **Check Tool Requirements:**\n   - The `Movie` function requires all four parameters: `title`, `year`, `director`, `ratings`\n   - I have all the required information.\n\n4.  **Construct Function Call:**\n   - `title`:

### Nested Sturcture

In [45]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name : str
    role : str
class MovieDetails(BaseModel):
    title : str
    year : int
    cast : list[str] = Field(description="List of main actors")
    genres : list[str]
    budget : float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails, method="json_schema")

response = model_with_structure.invoke("Provide details about the movie AngreziMedium in JSON Format")
response

BadRequestError: Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': ''}}

In [49]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field

# Make sure you have GEMINI_API_KEY set in your environment
# os.environ["GOOGLE_API_KEY"] = "your_key_here"

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

# Gemini handles Pydantic schemas natively without Groq's fragile JSON validator
model = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Tiger Zinda Hai")
print(response)

title='Tiger Zinda Hai' year=2017 cast=[Actor(name='Salman Khan', role='Tiger / Avinash Singh Rathore'), Actor(name='Katrina Kaif', role='Zoya'), Actor(name='Sajjad Delafrooz', role='Abu Usman'), Actor(name='Angad Bedi', role='Namit Khanna')] genres=['Action', 'Thriller', 'Adventure'] budget=32.0


### TypeDict

TypedDict provides a simpleer alternative using python's built-in typing, ideal when you don't need runtime validation 

In [51]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title : Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "the year the movie was released"]
    director: Annotated[str, "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_Typeddict = model.with_structured_output(MovieDict)
model_with_Typeddict.invoke("Please provide the details of the movie Angrezi Medium")

{'title': 'Angrezi Medium',
 'year': 2020,
 'director': 'Homi Adajania',
 'rating': 7.3}

In [53]:
class Actor(TypedDict):
    name : str
    role : str
class MovieDetails(TypedDict):
    title : str
    year : int
    cast : list[Actor]
    genres : list[str]
    budget : float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie AngreziMedium")
response

{'title': 'Angrezi Medium',
 'year': 2020,
 'cast': [{'name': 'Irrfan Khan', 'role': 'Champak Bansal'},
  {'name': 'Radhika Madan', 'role': 'Tarika Bansal'},
  {'name': 'Kareena Kapoor Khan', 'role': 'Naina Kohli'},
  {'name': 'Deepak Dobriyal', 'role': 'Gopi Bansal'}],
 'genres': ['Comedy', 'Drama'],
 'budget': 360000000}

In [55]:
model.profile #info of the model (what its supporting, what it does not support and all that......)

{'name': 'Gemini 3.5 Flash',
 'release_date': '2026-05-19',
 'last_updated': '2026-05-19',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True,
 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'],
 'reasoning_effort_default': 'medium'}

### DataClasses

A data class is a class typically containng mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator

In [57]:
import os 
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [63]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact info for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="Phone number of ther person")

agent = create_agent(
    model = "gpt-5",
    response_format=ContactInfo # Auto-selects providerStrategy
)

result = agent.invoke({
    "messages" : [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='d15df811-7b08-4791-985a-ce719b04652b'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1508, 'prompt_tokens': 203, 'total_tokens': 1711, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1472, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ENF5Jri6sXGPl3a21ttbhMl8EbnfT', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0951f-13f9-76c2-bbab-6569

In [64]:
print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [65]:
# TypedDict
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact info for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="Phone number of ther person")

agent = create_agent(
    model = "gpt-5",
    response_format=ContactInfo # Auto-selects providerStrategy
)

result = agent.invoke({
    "messages" : [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='db63b388-e1f9-4d9f-b236-b2c1a90d0733'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 484, 'prompt_tokens': 203, 'total_tokens': 687, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ENF6v9NDH55oBrgpc3QThoSVdpEOH', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a09520-9883-7e72-8020-1b801cd

In [66]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [67]:
## Data class
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    "contact details od person"
    name: str
    email: str
    phone: str

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo #Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages" : [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='58672c49-8b95-482a-a2fa-671a59f82033'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 292, 'prompt_tokens': 177, 'total_tokens': 469, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ENF9o0TVxTBFpnEx1hkg4xy8W8HM4', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a09523-55f6-7613-bd6d-07882f9

In [68]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')